# ZebraZoom: training a multi-channel (temporal) YOLO model

Each training image is made of several **video frames**, one per channel: the grayscale frames **n + frameOffsets** (for example
frames n-3, n and n+3), n being the labeled frame. The network can thus use the movement of the animals to detect them.

This dataset is created **on your computer** (from the Roboflow export and from your videos) with
[multiChannelYOLO_createDataset.py](https://github.com/oliviermirat/ZebraZoom/blob/master/DLrelatedFiles/multiChannelYOLO_createDataset.py),
which creates a zip file (option `--zip`).

1. Upload that zip file to your Google Drive.
2. Runtime > Change runtime type > T4 GPU (or any other GPU).
3. Set `zipPathOnGoogleDrive` in the second cell below.
4. Run all cells (you will be asked to allow access to your Google Drive). The last cell downloads the trained model
   (it's also copied next to the zip file on your Google Drive).

In [ ]:
!pip install -q ultralytics==8.4.183  # version with which this notebook was tested (multi-channel training needs ultralytics >= 8.3.1xx)

In [ ]:
# Parameters
zipPathOnGoogleDrive = '/content/drive/MyDrive/Cavefish-28_frames-3_0_+3.zip'  # zip created by multiChannelYOLO_createDataset.py
modelToStartFrom = 'yolo11n.pt'  # pretrained network (yolo11s.pt, yolo11m.pt, ... are bigger: more accurate but slower)
epochs = 1000
imgsz = 640  # if the dataset was created with --imageSize, imgsz must not be larger than that size
brightnessGain = 0.4  # random brightness changes of +/- 40% (same as the hsv_v=0.4 default of ultralytics' color augmentation)
averagePretrainedFirstConvOverChannels = True  # see the "Training" cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzipping the dataset
import os, zipfile
datasetsFolder = '/content/datasets'
with zipfile.ZipFile(zipPathOnGoogleDrive) as zipFile:
  dataYamlInZip = [name for name in zipFile.namelist() if name.endswith('data.yaml')][0]
  zipFile.extractall(datasetsFolder)
datasetFolder = os.path.join(datasetsFolder, os.path.dirname(dataYamlInZip))
print('Dataset unzipped in', datasetFolder)

In [ ]:
# Checking the dataset and setting its absolute path in data.yaml
import glob, yaml, cv2
from ultralytics.utils.patches import imread

dataYaml = os.path.join(datasetFolder, 'data.yaml')
with open(dataYaml) as f:
  dataConfig = yaml.safe_load(f)
dataConfig['path'] = datasetFolder
with open(dataYaml, 'w') as f:
  yaml.safe_dump(dataConfig, f, sort_keys=False)

nbChannels, frameOffsets = dataConfig['channels'], dataConfig['frameOffsets']
print('Channels:', nbChannels, '= grayscale frames', ', '.join('n%+d' % offset if offset else 'n' for offset in frameOffsets))
assert nbChannels == len(frameOffsets)
for split in ('train', 'val', 'test'):
  if split in dataConfig:
    images = glob.glob(os.path.join(datasetFolder, dataConfig[split], '*.tif'))
    labels = glob.glob(os.path.join(datasetFolder, dataConfig[split].replace('images', 'labels'), '*.txt'))
    print('%s: %d images, %d label files' % (split, len(images), len(labels)))
    image = imread(images[0], cv2.IMREAD_UNCHANGED)
    assert image.shape[2] == nbChannels, 'Wrong number of channels: ' + str(image.shape)
print('Image shape (height, width, channels):', image.shape)

In [ ]:
# Augmentations that keep the meaning of the channels (channels are video frames at different times, not colors)
import random
import numpy as np
import torch
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset

class SameBrightnessChangeOnAllChannels:
  """Random brightness change applied identically to every channel. It replaces ultralytics' HSV color augmentation
  (disabled below: on 3-channel images it would mix the frames, and it's not applied at all to images with more channels)."""
  def __init__(self, gain):
    self.gain = gain
  def __call__(self, labels):
    lookUpTable = np.clip(np.arange(256) * (1 + random.uniform(-self.gain, self.gain)), 0, 255).astype(np.uint8)
    labels['img'] = lookUpTable[labels['img']]
    return labels

if not hasattr(YOLODataset, 'originalBuildTransforms'):
  YOLODataset.originalBuildTransforms = YOLODataset.build_transforms
def buildTransforms(self, hyp=None):
  transforms = YOLODataset.originalBuildTransforms(self, hyp)
  if self.augment and brightnessGain:
    transforms.insert(len(transforms.transforms) - 1, SameBrightnessChangeOnAllChannels(brightnessGain))  # before the final Format transform
  return transforms
YOLODataset.build_transforms = buildTransforms

def initializeFirstConvolutionForFrameChannels(trainer):
  """All the channels of the first convolution get the pretrained filters summed over RGB, divided by the number of
  channels: on a still scene (all channels identical), the network then starts exactly as the pretrained network on a
  grayscale image. (Otherwise ultralytics gives the RGB filters to the first 3 channels and random filters to the others.)"""
  pretrainedWeights = YOLO(modelToStartFrom).model.model[0].conv.weight.detach().float()
  for model in (trainer.model, trainer.ema.ema if trainer.ema else None):
    if model is not None:
      convolution = (model.module if hasattr(model, 'module') else model).model[0].conv
      if convolution.weight.shape[0] == pretrainedWeights.shape[0] and convolution.weight.shape[2:] == pretrainedWeights.shape[2:]:
        with torch.no_grad():
          weights = pretrainedWeights.sum(1, keepdim=True).expand(-1, convolution.weight.shape[1], -1, -1) / convolution.weight.shape[1]
          convolution.weight.copy_(weights.to(convolution.weight.device, convolution.weight.dtype))
        print('First convolution initialized for', convolution.weight.shape[1], 'frame channels')

In [ ]:
# Training
from ultralytics import YOLO
model = YOLO(modelToStartFrom)
if averagePretrainedFirstConvOverChannels and modelToStartFrom.endswith('.pt'):
  model.add_callback('on_pretrain_routine_end', initializeFirstConvolutionForFrameChannels)

results = model.train(
  data=dataYaml,
  epochs=epochs,
  imgsz=imgsz,
  flipud=0.5,  # flips, mosaic, translation and scaling move all the channels (frames) identically, so they are kept
  fliplr=0.5,
  hsv_h=0.0,  # color augmentations disabled: they would mix the frames (replaced by SameBrightnessChangeOnAllChannels)
  hsv_s=0.0,
  hsv_v=0.0,
  augmentations=[],  # no Albumentations either (its default ToGray and CLAHE would also mix the frames of 3-channel images)
  bgr=0.0,  # never randomly swap the order of the channels (frames)
)

In [ ]:
metrics = results.results_dict
print(", ".join(f"{k.split('/')[-1]}: {v:.4f}" for k, v in metrics.items()))

In [ ]:
# Saving the best model in a cross-platform compatible format, with the frame offsets it was trained with
import shutil
from pathlib import Path

def convert_paths(obj):  # replaces PosixPath (not loadable on Windows) by strings
  if isinstance(obj, Path):
    return str(obj)
  elif isinstance(obj, dict):
    return {k: convert_paths(v) for k, v in obj.items()}
  elif isinstance(obj, list):
    return [convert_paths(i) for i in obj]
  elif isinstance(obj, tuple):
    return tuple(convert_paths(i) for i in obj)
  return obj

checkpoint = torch.load(model.trainer.best, map_location='cpu', weights_only=False)
checkpoint = convert_paths(checkpoint)
checkpoint['zebrazoomFrameOffsets'] = frameOffsets  # checked by ZebraZoom against "yoloMultiChannelFrameOffsets"
modelFileName = 'model_frames' + '_'.join('%+d' % offset if offset else '0' for offset in frameOffsets) + '_crossPlatform_compatible.pt'
torch.save(checkpoint, modelFileName)
print('Model saved:', modelFileName)
print('In the ZebraZoom configuration file, use:\n  "DLmodelPath": "%s",\n  "yoloMultiChannelFrameOffsets": %s' % (modelFileName, [offset for offset in frameOffsets if offset]))

In [ ]:
shutil.copy(modelFileName, os.path.dirname(zipPathOnGoogleDrive))  # copy on Google Drive, next to the zip file
from google.colab import files
files.download(modelFileName)